In [ ]:
# ==============================================================================
# SPAS WORKSHOP: NON-PARAMETRIC STATISTICAL ANALYSIS IN PYTHON
# Dataset: SPAS_Conference_Training_Data.csv
# ==============================================================================

import pandas as pd
import numpy as np
import scipy.stats as stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# 1. Load Dataset
url = "https://raw.githubusercontent.com/softdataconsult/SPAS-Data-Analytics-Workshop-2026/main/SPAS_Conference_Training_Data.csv"
df = pd.read_csv(url)

# ------------------------------------------------------------------------------
# EXERCISE 1: MEDIAN & INTERQUARTILE RANGE (Descriptive Non-Parametrics)
# ------------------------------------------------------------------------------
descriptives = df.groupby('Department')['Enzyme_Activity'].agg(
    Median='median',
    Q25=lambda x: x.quantile(0.25),
    Q75=lambda x: x.quantile(0.75),
    IQR=lambda x: x.quantile(0.75) - x.quantile(0.25)
)
print("--- Non-Parametric Summary Statistics (Median & IQR) ---")
print(descriptives)

# ------------------------------------------------------------------------------
# EXERCISE 2: TWO INDEPENDENT GROUPS (Mann-Whitney U Test)
# ------------------------------------------------------------------------------
pkg_a = df[df["Packaging_Type"] == "Packaging_A"]["Shelf_Life_Days"]
pkg_b = df[df["Packaging_Type"] == "Packaging_B"]["Shelf_Life_Days"]

u_stat, u_pvalue = stats.mannwhitneyu(pkg_a, pkg_b, alternative='two-sided')
print(f"\nMann-Whitney U Test: U = {u_stat:.2f}, p-value = {u_pvalue:.4e}")

# Visualization
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Packaging_Type", y="Shelf_Life_Days", palette="Blues")
sns.stripplot(data=df, x="Packaging_Type", y="Shelf_Life_Days", color="black", alpha=0.5)
plt.title("Shelf Life Comparison (Mann-Whitney U Test)")
plt.xlabel("Packaging Type")
plt.ylabel("Shelf Life (Days)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# EXERCISE 3: THREE+ INDEPENDENT GROUPS (Kruskal-Wallis Test & Dunn's Post-Hoc)
# ------------------------------------------------------------------------------
dept_groups = [group["Enzyme_Activity"].values for name, group in df.groupby("Department")]

kw_stat, kw_pvalue = stats.kruskal(*dept_groups)
print(f"\nKruskal-Wallis H Test: H = {kw_stat:.4f}, p-value = {kw_pvalue:.4e}")

# Dunn's Post-Hoc Test with Bonferroni Correction
dunn_posthoc = sp.posthoc_dunn(df, val_col='Enzyme_Activity', group_col='Department', p_adjust='bonferroni')
print("\n--- Dunn's Post-Hoc Test (Pairwise p-values) ---")
print(dunn_posthoc)

# Visualization
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Department", y="Enzyme_Activity", palette="Set2")
plt.title("Enzyme Activity Across Departments (Kruskal-Wallis Test)")
plt.xlabel("Department")
plt.ylabel("Enzyme Activity (U/mL)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# EXERCISE 4: MONOTONIC RELATIONSHIP (Spearman Rank Correlation)
# ------------------------------------------------------------------------------
rho, rho_pvalue = stats.spearmanr(df["Temperature_C"], df["Yield_Pct"])
print(f"\nSpearman Rank Correlation: rho = {rho:.4f}, p-value = {rho_pvalue:.4e}")

# Visualization with Non-Parametric LOESS curve
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x="Temperature_C", y="Yield_Pct", color="#003366", lowess=True, line_kws={"color": "red"})
plt.title(f"Spearman Monotonic Association: Yield vs Temp (rho = {rho:.3f})")
plt.xlabel("Temperature (°C)")
plt.ylabel("Yield (%)")
plt.tight_layout()
plt.show()